In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import balanced_accuracy_score

print("🚀 MIVIA DAY 3: Transfer Learning con MobileNetV2 🚀\n")

OUTPUT_CSV = '../dataset/train_val_split.csv'

#carichiamo il dataset
try:
    df = pd.read_csv(OUTPUT_CSV)
    train_df = df[df['split'] == 'train']
    val_df = df[df['split'] == 'val']
    print(f" CSV caricato: {len(train_df)} immagini di Train, {len(val_df)} di Validation.")
except FileNotFoundError:
    raise FileNotFoundError(f" ERRORE: Non trovo il file {OUTPUT_CSV}.")

#data loader
print("\n--- FASE 2: Preparazione Tensori ---")

base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self): return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['filepath']
        image = Image.open(img_path).convert('RGB')
        label = self.dataframe.iloc[idx]['label']
        if self.transform: image = self.transform(image)
        return image, label

BATCH_SIZE = 32 
train_loader = DataLoader(WasteDataset(train_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(WasteDataset(val_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=False)
print(" Dataloader pronti!")


#TRANSFER LEARNING (MobileNetV2)
print("\n--- FASE 3: Inizializzazione MobileNetV2 ---")


weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)


for param in model.parameters():
    param.requires_grad = False


num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 8)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

print(f" MobileNetV2 pronta. Parametri congelati, testa modificata per 8 classi.")
print(f" Acceleratore in uso: {device.type.upper()}")

#addestramento e accuracy
print("\n--- FASE 4: Addestramento ---")

EPOCHS = 5 

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if i % 20 == 19: 
            print(f"  Epoca [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {running_loss/20:.4f}")
            running_loss = 0.0
            
    print(f" Epoca {epoch+1} conclusa. Calcolo metriche di validazione...")
    
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Calcolo della Balanced Accuracy richiesta dal progetto
    bal_acc = balanced_accuracy_score(all_labels, all_preds) * 100
    print(f"🏆 Balanced Accuracy all'epoca {epoch+1}: {bal_acc:.2f}%\n")

torch.save(model.state_dict(), 'mobilenet_mivia_weights.pth')
print("🎉 Addestramento concluso! Modello salvato in: mobilenet_mivia_weights.pth")

🚀 MIVIA DAY 3: Transfer Learning con MobileNetV2 🚀

 CSV caricato: 12412 immagini di Train, 3103 di Validation.

--- FASE 2: Preparazione Tensori ---
 Dataloader pronti!

--- FASE 3: Inizializzazione MobileNetV2 Avanzata ---
🔓 Ultimo blocco convoluzionale (Block 18) sbloccato per il fine-tuning!
✅ Modello modificato pronto con Testa Multi-Layer.
✅ Acceleratore in uso: MPS

--- FASE 4: Addestramento ---
  Epoca [1/5], Batch [20/388], Loss: 1.0879
  Epoca [1/5], Batch [40/388], Loss: 0.5913
  Epoca [1/5], Batch [60/388], Loss: 0.4834
  Epoca [1/5], Batch [80/388], Loss: 0.4125
  Epoca [1/5], Batch [100/388], Loss: 0.3904
  Epoca [1/5], Batch [120/388], Loss: 0.3628
  Epoca [1/5], Batch [140/388], Loss: 0.3301
  Epoca [1/5], Batch [160/388], Loss: 0.3406
  Epoca [1/5], Batch [180/388], Loss: 0.3228
  Epoca [1/5], Batch [200/388], Loss: 0.3332
  Epoca [1/5], Batch [220/388], Loss: 0.2813
  Epoca [1/5], Batch [240/388], Loss: 0.2532
  Epoca [1/5], Batch [260/388], Loss: 0.3201
  Epoca [1/5]

KeyboardInterrupt: 